In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone --depth 1 -q https://github.com/khangnghiem/deep-learning.git /content/deep-learning
import os, sys, glob, json
sys.path.insert(0, '/content/deep-learning')

EXPERIMENT = '017_polyp_rtdetr'
REPO = 'deep-learning'

print('=' * 60)
print(f'🧪 E2E TEST: {EXPERIMENT}')
print('=' * 60)

In [ ]:
# ✅ 1/5 — Verify data exists
data_path = '/content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset'
assert os.path.exists(data_path), f'❌ Data not found: {data_path}'
assert os.path.exists(os.path.join(data_path, 'dataset.yaml')), '❌ dataset.yaml missing'
items = os.listdir(data_path)
print(f'✅ [1/5] Data verified: {len(items)} items at {data_path}')

In [ ]:
# ✅ 2/5 — Verify experiment files
exp_path = f'/content/{REPO}/experiments/{EXPERIMENT}'
required = ['config.yaml', 'train.py', '017_polyp_rtdetr_train.ipynb', 'README.md']
for f in required:
    assert os.path.exists(os.path.join(exp_path, f)), f'❌ Missing: {f}'
    print(f'  📄 {f}')
print(f'✅ [2/5] All experiment files present')

In [ ]:
# ✅ 3/5 — Verify trained model exists
model_patterns = [
    f'/content/drive/MyDrive/models/trained/{EXPERIMENT}/**/*.pt',
    f'/content/drive/MyDrive/models/trained/{EXPERIMENT}/train/weights/*.pt',
]
found = []
for p in model_patterns:
    found.extend(glob.glob(p, recursive=True))
assert len(found) > 0, '❌ No trained model found'
for m in found[:5]:
    size_mb = os.path.getsize(m) / 1024 / 1024
    print(f'  📦 {os.path.basename(m)} ({size_mb:.1f} MB)')
print(f'✅ [3/5] Trained model: {len(found)} file(s)')

In [ ]:
# ✅ 4/5 — Verify MLflow run
import mlflow
mlflow.set_tracking_uri('file:///content/drive/MyDrive/ops/mlflow/mlruns')
runs = mlflow.search_runs(experiment_names=['polyp-rtdetr'], max_results=5)
assert len(runs) > 0, f'❌ No MLflow runs for polyp-rtdetr'
best = runs.iloc[0]
for col in runs.columns:
    if col.startswith('metrics.'):
        val = best[col]
        if val == val:  # skip NaN
            print(f'  📊 {col.replace("metrics.", "")}: {val}')
print(f'✅ [4/5] MLflow run: {best["run_id"][:8]}...')

In [ ]:
# ✅ 5/5 — Verify model inference
from ultralytics import RTDETR
import torch

best_pt = glob.glob(f'/content/drive/MyDrive/models/trained/{EXPERIMENT}/**/best.pt', recursive=True)
assert len(best_pt) > 0, '❌ best.pt not found'

model = RTDETR(best_pt[0])
# Run inference on a val image
val_imgs = glob.glob('/content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset/val/images/*.*')
assert len(val_imgs) > 0, '❌ No val images'

results = model.predict(val_imgs[0], conf=0.25, verbose=False)
n_boxes = len(results[0].boxes) if len(results) > 0 else 0
print(f'  Detected {n_boxes} polyp(s) in sample image')
print(f'✅ [5/5] Inference test passed')

In [ ]:
print('\n' + '=' * 60)
print('🏁 E2E TEST COMPLETE')
print('=' * 60)
checks = [
    'Data exists and is non-empty',
    'Experiment files are complete',
    'Trained model checkpoint exists',
    'MLflow tracking has logged runs',
    'Model inference produces valid output',
]
for i, c in enumerate(checks, 1):
    print(f'  ✅ {i}. {c}')
print(f'\n🎉 ALL {len(checks)} CHECKS PASSED')
print('=' * 60)

In [ ]:
from google.colab import runtime
runtime.unassign()
